# Number Theory

A toolkit of the math that turns up constantly in algorithm problems — GCD, primes, modular arithmetic, combinatorics — and the Python built-ins (`math.gcd`, `pow(a, b, m)`, `math.comb`) that implement most of it for you. The angle, as ever: know the algorithm, but reach for the C-level builtin in practice.

**Contents**

1. **GCD & LCM** — Euclid's algorithm
2. **Primes** — the sieve of Eratosthenes
3. **Primality testing** — Miller–Rabin
4. **Modular arithmetic** — fast exponentiation & inverse
5. **Combinatorics**
6. **Toolkit & cheat-sheet**


## 1. GCD & LCM — Euclid's algorithm

The **greatest common divisor** comes from Euclid's ~2300-year-old insight: `gcd(a, b) = gcd(b, a mod b)`, repeated until the remainder is 0 — O(log min(a, b)). LCM follows from `lcm(a, b) = a * b // gcd(a, b)` (divide first to keep the intermediate small):

In [1]:
def gcd(a, b):
    while b:
        a, b = b, a % b                      # Euclid: replace (a, b) with (b, a mod b)
    return a

def lcm(a, b):
    return a // gcd(a, b) * b

print("gcd(48, 18):", gcd(48, 18))
print("lcm(4, 6)  :", lcm(4, 6))

import math
print("math.gcd / math.lcm:", math.gcd(48, 18), math.lcm(4, 6))   # the builtins (variadic)


gcd(48, 18): 6
lcm(4, 6)  : 12
math.gcd / math.lcm: 6 12


`math.gcd` and `math.lcm` (3.9+) take any number of arguments. The **extended** Euclidean algorithm also finds `x, y` with `ax + by = gcd(a, b)` — the basis of the modular inverse in §3.

## 2. Primes — the sieve of Eratosthenes

To list every prime up to n, the **sieve** beats trial-dividing each number. Mark all numbers "prime," then for each prime p cross off its multiples starting at `p²` (smaller multiples are already gone). It's **O(n log log n)** — effectively linear:

In [2]:
def sieve(n):
    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False
    for i in range(2, int(n ** 0.5) + 1):
        if is_prime[i]:
            for j in range(i * i, n + 1, i):  # start at i^2; mark multiples composite
                is_prime[j] = False
    return [i for i in range(n + 1) if is_prime[i]]

print("primes <= 30:", sieve(30))


primes <= 30: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


To test a *single* number for primality, trial division up to √n is O(√n); for very large numbers the practical choice is **Miller-Rabin** (probabilistic).

## 3. Primality testing — Miller–Rabin

The sieve gives *all* primes up to n, but to test a single large number, trial division costs O(√n) — hopeless for 60-digit inputs. **Miller–Rabin** is the practical answer: write n−1 = 2^s·d with d odd, then for a base `a` check whether `a^d ≡ 1 (mod n)` or `a^(d·2^r) ≡ −1 (mod n)` for some r < s. A prime passes for *every* base; a composite fails for at least ¾ of bases, so k random bases give a 4^(−k) error bound. With a fixed set of small-prime bases the test is **deterministic** below well-known thresholds. It leans on the fast modular exponentiation of the next section — here via the C-level `pow(a, d, n)`.

In [3]:
def miller_rabin(n, bases=(2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)):
    """Deterministic for n < 3.3 x 10^24 with these 12 bases."""
    if n < 2:
        return False
    for p in bases:
        if n % p == 0:
            return n == p                       # divisible by a small prime
    d, s = n - 1, 0
    while d % 2 == 0:                            # n - 1 = 2^s * d, d odd
        d //= 2; s += 1
    for a in bases:
        x = pow(a, d, n)                        # C-level fast modexp
        if x == 1 or x == n - 1:
            continue
        for _ in range(s - 1):
            x = x * x % n
            if x == n - 1:
                break
        else:
            return False                        # witness a proves n composite
    return True

# cross-check against a sieve for every n up to 10000
def _sieve_set(n):
    sieve = bytearray([1]) * (n + 1)
    sieve[0] = sieve[1] = 0
    for i in range(2, int(n ** 0.5) + 1):
        if sieve[i]:
            sieve[i * i::i] = bytearray(len(sieve[i * i::i]))
    return sieve

N = 10000
sv = _sieve_set(N)
assert all(miller_rabin(n) == bool(sv[n]) for n in range(N + 1))
print("Miller-Rabin matches the sieve for every n <= 10000: OK")

# and it instantly handles numbers far beyond any sieve:
m89 = 2 ** 89 - 1                               # a Mersenne prime
print(f"2^89 - 1  prime?     {miller_rabin(m89)}   ({m89})")
print(f"2^89 + 1  composite? {not miller_rabin(2 ** 89 + 1)}   (divisible by 3)")


Miller-Rabin matches the sieve for every n <= 10000: OK
2^89 - 1  prime?     True   (618970019642690137449562111)
2^89 + 1  composite? True   (divisible by 3)


## 4. Modular arithmetic — fast exponentiation & inverse

Computations "mod m" (cryptography, hashing, counting answers mod 10⁹+7) need two tools.

**Modular exponentiation** computes `base^exp mod m` without ever building the giant `base^exp`, by **squaring** — O(log exp). Python's three-argument `pow` does exactly this in C. **Modular inverse** is the "division" of modular arithmetic — `x` with `a·x ≡ 1 (mod m)`; since Python 3.8, `pow(a, -1, m)` gives it directly (it exists iff `gcd(a, m) == 1`):

In [4]:
def mod_pow(base, exp, mod):
    result = 1
    base %= mod
    while exp:
        if exp & 1:                          # bit set -> fold the current square in
            result = result * base % mod
        base = base * base % mod             # square the base
        exp >>= 1
    return result

print("mod_pow(2, 10, 1000)   :", mod_pow(2, 10, 1000))
print("builtin pow(2, 10, 1000):", pow(2, 10, 1000))   # same thing, in C - just use this

# modular inverse (Python 3.8+):
inv = pow(3, -1, 7)
print("inverse of 3 mod 7     :", inv, "-> check 3 * inv % 7 =", 3 * inv % 7)


mod_pow(2, 10, 1000)   : 24
builtin pow(2, 10, 1000): 24
inverse of 3 mod 7     : 5 -> check 3 * inv % 7 = 1


The inverse lets you "divide" under a modulus — essential for `C(n, k) mod p` and other counting-mod-prime problems. By Fermat's little theorem, for prime `p` the inverse of `a` is also `pow(a, p - 2, p)`.

## 5. Combinatorics

Counting arrangements — permutations, combinations, factorials — has direct `math` builtins, all **exact** thanks to arbitrary-precision ints (bit-manipulation notebook):

In [5]:
import math
print("factorial(5):", math.factorial(5))   # 120
print("comb(5, 2)  :", math.comb(5, 2))      # 10  = C(5,2)
print("perm(5, 2)  :", math.perm(5, 2))      # 20  = 5*4
print("comb(52, 5) :", math.comb(52, 5))     # poker hands, computed exactly


factorial(5): 120
comb(5, 2)  : 10
perm(5, 2)  : 20
comb(52, 5) : 2598960


## 6. Toolkit & cheat-sheet

**Python toolkit:** `math.gcd` / `math.lcm` (variadic), `pow(base, exp, mod)` (fast modexp), `pow(a, -1, m)` (modular inverse, 3.8+), `math.comb` / `math.perm` / `math.factorial`, and `math.isqrt` (integer sqrt — the searching notebook). Arbitrary-precision `int` means every one of these is **exact**, never overflowing.

| Task | Algorithm | Complexity | Builtin |
|---|---|:---:|---|
| GCD / LCM | Euclid | O(log n) | `math.gcd` / `math.lcm` |
| All primes ≤ n | sieve of Eratosthenes | O(n log log n) | — |
| Single primality | trial division / Miller-Rabin | O(√n) / prob. | — |
| base^exp mod m | binary exponentiation | O(log exp) | `pow(b, e, m)` |
| modular inverse | extended Euclid / Fermat | O(log m) | `pow(a, -1, m)` |
| nCk, nPk, n! | — | — | `math.comb` / `perm` / `factorial` |